# 15 - Analogo en R (recreado desde cero)

## Tema base

**15 - Iteradores y generadores: eficiencia y control fino del flujo**


## Objetivos de la sesion en R

1. Mantener la teoria clave del notebook Python en equivalente R.
2. Usar solo codigo ejecutable en R.
3. Conservar ejercicios adaptados a R.
4. Agregar probabilidad y estadistica acorde al tema.


## Ruta Teoria-Codigo (alternada)

Se organiza la sesion en bloques cortos que alternan concepto y practica.


### Bloque teorico 1

### Teoria 1

# 15 - Iteradores y generadores: eficiencia y control fino del flujo

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Distinguir claramente entre `iterable` e `iterador`.
2. Entender el protocolo de iteracion (`__iter__`, `__next__`, `StopIteration`).
3. Explicar que hace un `for` internamente.
4. Construir iteradores personalizados con clases.
5. Crear generadores con `yield` y razonar sobre su estado.
6. Usar expresiones generadoras para ahorrar memoria.
7. Encadenar transformaciones de forma lazy (perezosa).
8. Aplicar `itertools` para resolver tareas comunes sin reinventar la rueda.
9. Identificar errores sutiles: agotamiento, consumo accidental y efectos secundarios.
10. Resolver problemas donde importa mas el flujo de datos que la sintaxis.

### Teoria 2

## 1. Modelo mental: iterable vs iterador

Un **iterable** es cualquier objeto que puede producir elementos uno por uno.
Ejemplos: `list`, `tuple`, `str`, `dict`, `set`, `range`, archivos abiertos.

Un **iterador** es el objeto que realmente lleva la cuenta del avance.
- Tiene `__next__()` para entregar el siguiente elemento.
- Cuando no hay mas, lanza `StopIteration`.

Regla practica:
- Iterable = contenedor o fuente.
- Iterador = cursor que avanza sobre esa fuente.

### Teoria 3

## 2. Protocolo de iteracion en detalle

R usa un contrato simple:

1. `iter(obj)` intenta obtener un iterador desde `obj.__iter__()`.
2. `next(it)` llama `it.__next__()` para pedir el siguiente valor.
3. Si el iterador termino, `__next__()` debe lanzar `StopIteration`.

Este contrato hace que muchos componentes distintos se puedan combinar:
`for`, comprensiones, `sum`, `min`, `max`, `list`, `tuple`, `set`, `dict`, etc.

### Teoria 4

## 3. Que hace `for` internamente

`for x in algo:` no "adivina" nada. Sigue el protocolo anterior.
Equivale conceptualmente a:

1. Crear iterador con `iter(algo)`.
2. Pedir elementos con `next(...)` en bucle.
3. Detenerse al recibir `StopIteration`.

Entender esto ayuda a depurar errores cuando mezclas loops, funciones y generadores.

### Teoria 5

## 4. Iteradores personalizados con clases

Crear una clase iteradora te da control explicito sobre:
- Estado interno.
- Condicion de parada.
- Regla para producir cada siguiente valor.

Es util cuando el flujo necesita varias variables de estado o reglas complejas.


In [ ]:
crear_contador <- function(inicio=0){i <- inicio; function(){i <<- i+1; i}}
cont <- crear_contador(10)
cont(); cont(); cont()


### Bloque teorico 2

### Teoria 6

## 5. Generadores con `yield`

Un generador permite escribir logica de iteracion sin definir una clase completa.

Cada vez que aparece `yield`:
- Se devuelve un valor al consumidor.
- La funcion queda "pausada".
- Al siguiente `next(...)`, se reanuda desde ese punto.

Ventaja clave: codigo compacto y natural para flujos secuenciales.

### Teoria 7

## 6. El estado vive dentro del generador

Un generador recuerda variables locales entre llamadas.
Eso evita usar estado global o clases cuando no hace falta.

Piensa en un generador como una maquina de estados pequena.

### Teoria 8

## 7. `return` dentro de un generador

En un generador, `return` no entrega un valor normal al `for`.
Lo que hace es terminar la iteracion lanzando `StopIteration`.

En usos avanzados (consumo manual), ese `return` puede leerse desde `StopIteration.value`.
No es necesario para tareas basicas, pero entenderlo evita confusion.

### Teoria 9

## 8. Comprension de listas vs expresion generadora

Diferencia central:
- `[f(x) for x in datos]` crea toda la lista de una vez.
- `(f(x) for x in datos)` produce valores bajo demanda.

Si solo vas a recorrer una vez y los datos son grandes, el generador suele ser mejor.
Si necesitas acceso aleatorio o reutilizar resultados muchas veces, la lista puede convenir.

### Teoria 10

## 9. Pipelines lazy: transformar sin cargar todo en memoria

Puedes encadenar pasos para limpiar, filtrar y transformar datos de forma incremental.
Cada paso consume y produce iteradores.

Beneficios:
- Menor memoria.
- Flujo mas claro.
- Mejor composicion de funciones pequenas.


In [ ]:
crear_fib <- function(){a <- 0; b <- 1; function(){out <- a; tmp <- a+b; a <<- b; b <<- tmp; out}}
f <- crear_fib()
replicate(10, f())


### Bloque teorico 3

### Teoria 11

## 10. Lectura de archivos: iterar linea por linea

En archivos grandes, evita `read()` completo cuando no es necesario.
Iterar por lineas permite procesar streaming y cortar temprano si ya tienes lo que buscabas.

### Teoria 12

## 11. `itertools`: herramientas de alto rendimiento

`itertools` trae bloques listos para trabajar con iteradores.
Algunos muy utiles:
- `islice`: cortar sin materializar todo.
- `chain`: concatenar iterables.
- `count`: contador infinito.
- `takewhile`: tomar mientras una condicion se cumpla.

Usarlos reduce codigo manual y errores.

### Teoria 13

## 12. Cuando usar clase iteradora y cuando generador

Usa **generador** cuando:
- La secuencia se expresa bien de forma lineal.
- Quieres menos codigo boilerplate.

Usa **clase iteradora** cuando:
- Necesitas multiples metodos y control detallado del estado.
- Quieres exponer comportamiento adicional (reset, estadisticas, configuracion).

No existe "mejor siempre"; depende del problema.

### Teoria 14

## 13. Errores comunes y buenas practicas

Errores tipicos:
1. Consumir un iterador y luego asumir que aun tiene datos.
2. Convertir a `list(...)` demasiado pronto (rompe el beneficio lazy).
3. Ocultar efectos secundarios dentro de generadores (dificulta depuracion).
4. Usar nombres ambiguos (`it`, `x`) en pipelines largos.

Buenas practicas:
1. Documenta si una funcion devuelve iterable reutilizable o iterador de un solo uso.
2. Separa pasos del pipeline en funciones cortas con nombres claros.
3. Prueba casos borde: vacio, un elemento, y datos invalidos.
4. Cuando el flujo sea complejo, agrega asserts o logs pequenos.

### Teoria 15

## 15. Resumen de conceptos clave

1. Iterable no es lo mismo que iterador.
2. `for` depende del protocolo `iter`/`next`.
3. `yield` permite modelar flujos de datos con estado y pausa.
4. Expresiones generadoras y `itertools` ayudan a escalar procesamiento.
5. Pensar en consumo, agotamiento y composicion evita bugs sutiles.


In [ ]:
crear_batches <- function(x, batch_size=3){i <- 1; n <- length(x); function(){if(i>n) return(NULL); j <- min(i+batch_size-1,n); out <- x[i:j]; i <<- j+1; out}}
it <- crear_batches(1:10,4)
it(); it(); it(); it()


## Extra de probabilidad y estadistica

Media acumulada de secuencias pseudoaleatorias y convergencia.


In [ ]:
set.seed(2026)
vals <- runif(5000)
media_acum <- cumsum(vals)/seq_along(vals)
plot(media_acum, type='l', col='steelblue'); abline(h=0.5, col='red', lty=2)


## Analogias R y Python

- Generadores con estado en Python se emulan con closures en R.
- Iteracion por lotes es patron comun en ambos.


In [ ]:
# Checklist rapido R vs Python
# 1) ?Que cambia en sintaxis?
# 2) ?Que cambia en estructura de datos?
# 3) ?Que cambia en manejo de NA/errores?


## Errores comunes

- Estado global accidental (`<<-` mal usado).
- No definir condicion de paro.
- Consumir iterador sin control de memoria.


In [ ]:
# Auto-verificacion de errores comunes
# Define un caso borde y valida que tu solucion en R no falle silenciosamente.


## Ejercicios para pensar (no copiar codigo)

Primero argumenta en texto. Luego, si hace falta, valida con experimentos en R.


### Ejercicio reflexivo 1

### Ejercicio 1

## 14. Ejercicios de razonamiento cuidadoso

Los siguientes ejercicios estan pensados para obligarte a:
- Razonar sobre estado y consumo.
- Elegir entre materializar vs procesar lazy.
- Detectar bugs que no son de sintaxis, sino de flujo.

Sugerencia: intenta primero sin buscar la solucion.


### Ejercicio reflexivo 2

### Ejercicio 2

### Ejercicio 1: De lista a generador sin perder claridad

**Tarea**: Reescribe `cuadrados_pares_lista` como `cuadrados_pares_gen`.

Condiciones:
1. Debe devolver un generador, no lista.
2. Debe procesar bajo demanda.
3. Mantener legibilidad.


### Ejercicio reflexivo 3

### Ejercicio 3

### Ejercicio 2: Iterador por bloques

**Tarea**: Implementa una clase `Bloques` que reciba una secuencia y un tamano de bloque.
Debe iterar devolviendo sublistas de longitud `k` (la ultima puede ser menor).

Ejemplo:
- Entrada: `[1,2,3,4,5]`, `k=2`
- Salida: `[1,2]`, `[3,4]`, `[5]`

Piensa con cuidado como detenerte exactamente cuando corresponde.


### Ejercicio reflexivo 4

### Ejercicio 4

### Ejercicio 3: Agotamiento silencioso

**Tarea**: Explica por que este codigo imprime una lista vacia en la segunda conversion.
Luego corrige el flujo sin duplicar trabajo innecesario.


### Ejercicio reflexivo 5

### Ejercicio 5

### Ejercicio 4: Depurar un pipeline

**Tarea**: Hay un bug logico: el pipeline intenta quedarse con lineas validas,
pero termina descartando casi todo. Detecta el error y corrigelo.

Pista: revisa el orden de `strip`, `split` y la condicion.


### Ejercicio reflexivo 6

### Ejercicio 6

### Ejercicio 5: Merge lazy de dos secuencias ordenadas

**Tarea**: Implementa `merge_ordenado(a, b)` que mezcle dos iterables ordenados
sin materializar ambos por completo.

Requisitos:
1. Debe producir elementos en orden ascendente.
2. Debe funcionar aunque uno termine antes.
3. Debe aceptar iterables (no solo listas).


### Ejercicio reflexivo 7

### Ejercicio 7

### Ejercicio 6: Ventana deslizante

**Tarea**: Crea un generador `ventanas(iterable, k)` que produzca tuplas de tamano `k`
con ventanas consecutivas.

Ejemplo con `k=3` y `[1,2,3,4,5]`:
- `(1,2,3)`, `(2,3,4)`, `(3,4,5)`

Piensa bien que hacer cuando `k` es mayor que la cantidad de datos.


### Ejercicio reflexivo 8

### Ejercicio 8

### Ejercicio 7: Flatten recursivo con control de strings

**Tarea**: Implementa `aplanar(datos)` para aplanar listas/tuplas anidadas,
pero tratando strings como atomicos (no iterar caracter por caracter).

Ejemplo:
`[1, [2, (3, 4)], "abc", ["xy", [5]]]` -> `1,2,3,4,"abc","xy",5`


### Ejercicio reflexivo 9

### Ejercicio 9

### Ejercicio 8: CSV lazy con validacion

**Tarea**: Dado un iterable de lineas CSV `id,monto`, crea un generador
`leer_montos_validos(lineas)` que entregue solo montos positivos validos (`float`).

Ademas, registra en una lista externa `errores` las lineas rechazadas con su motivo.

Este ejercicio obliga a separar claramente:
1. Parseo.
2. Validacion.
3. Flujo de salida.


### Ejercicio reflexivo 10

### Ejercicio 10

### Ejercicio 9: Round-robin justo

**Tarea**: Implementa `round_robin(*iterables)` que intercale elementos de cada iterable
sin bloquearse cuando uno se agota.

Ejemplo:
`[1,2,3]`, `"AB"`, `[10,20,30,40]` -> `1,"A",10,2,"B",20,3,30,40`

Punto clave: eliminar de forma segura los iteradores ya agotados.


### Ejercicio reflexivo 11

### Ejercicio 11

### Ejercicio 10: Diseno y justificacion tecnica

**Tarea**: Tienes 20 millones de registros de texto y solo necesitas:
1. Contar cuantas lineas cumplen una condicion.
2. Obtener las primeras 50 que fallan validacion.

Escribe dos soluciones:
- Enfoque A materializando en listas.
- Enfoque B lazy con iteradores/generadores.

Luego compara tiempo/memoria de forma razonada y argumenta cual usarias en produccion.


### Ejercicio reflexivo 12

### Ejercicio 12

## 16. Reto final opcional

Implementa un mini ETL lazy:
1. Lee lineas simuladas de un log.
2. Filtra solo eventos `VENTA` validos.
3. Convierte montos a `float`.
4. Calcula total, promedio y top 3 montos sin guardar todo si no es necesario.

Objetivo: entrenar el criterio para elegir estructuras de datos y estrategia de iteracion.


### Preguntas de cierre

1. ?Que supuesto estadistico es mas fragil en esta clase?
2. ?Que evidencia minima pedirias antes de usar este enfoque en un caso real?
3. ?Como explicarias este tema a alguien no tecnico sin perder rigor?
4. ?Que cambiaria entre una implementacion en R y una en Python para produccion?
